# Study 01 — How often should a penalty shootout reach sudden death?

## Question

Under a deliberately simple penalty-shootout model, how often should the shootout still be level after the initial five kicks per team and therefore continue into sudden death?

This study is inspired by the penalty-shootout example in Joe Buchdahl's *Monte Carlo Or Bust*, but the model and implementation here are built independently.

## Rules and scope

The current IFAB Laws of the Game specify that, subject to early termination when one team can no longer catch the other, both teams take five alternating kicks. If the scores are level after five kicks each, kicks continue until one team leads after the same number of kicks.

For this first model we assume:

- the two teams are equally strong;
- every penalty is independent;
- every penalty has the same scoring probability, `p`;
- there are no kicker, goalkeeper, order, pressure or score-state effects;
- we study only whether sudden death is reached, not how long sudden death lasts or who wins.

Under these assumptions, reaching sudden death is equivalent to the two teams scoring the same number of goals from their five scheduled kicks.

**External rule source:** The IFAB, Law 10 — *Determining the Outcome of a Match*, Laws of the Game 2026/27.

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 20260817

## Model

Each team's number of successful penalties in the first five kicks is a binomial random variable:

\[
X \sim \mathrm{Binomial}(5, p), \qquad Y \sim \mathrm{Binomial}(5, p)
\]

with `X` and `Y` independent. Sudden death is reached when `X == Y`.

That gives an exact reference probability:

\[
P(\text{sudden death}) = \sum_{k=0}^{5} P(X=k)P(Y=k)
= \sum_{k=0}^{5} \left[{5 \choose k}p^k(1-p)^{5-k}\right]^2.
\]

The exact calculation is useful as a validation target; the simulation itself simply draws the two five-kick totals and checks whether they are equal.

In [ ]:
def exact_sudden_death_probability(p: float) -> float:
    """Exact probability that two independent Binomial(5, p) totals are equal."""
    return sum(
        (comb(5, k) * p**k * (1 - p) ** (5 - k)) ** 2
        for k in range(6)
    )


def simulate_sudden_death_probability(
    p: float,
    n_shootouts: int,
    rng: np.random.Generator,
) -> float:
    """Monte Carlo estimate of the probability of reaching sudden death."""
    scores = rng.binomial(n=5, p=p, size=(n_shootouts, 2))
    return np.mean(scores[:, 0] == scores[:, 1])

## Validation against the 72.5% benchmark

The benchmark supplied for this study is a constant penalty scoring probability of **72.5%**, with a stated exact probability of reaching sudden death of **27.987%**.

We first reproduce the exact result independently from the formula above, then run a large Monte Carlo simulation. A fixed random seed makes the result reproducible.

In [ ]:
benchmark_p = 0.725
benchmark_n = 2_000_000
benchmark_rng = np.random.default_rng(SEED)

benchmark_sim = simulate_sudden_death_probability(
    benchmark_p,
    benchmark_n,
    benchmark_rng,
)
benchmark_exact = exact_sudden_death_probability(benchmark_p)
benchmark_se = np.sqrt(benchmark_sim * (1 - benchmark_sim) / benchmark_n)
benchmark_half_width = 1.96 * benchmark_se

benchmark = pd.DataFrame(
    {
        "scoring_probability": [benchmark_p],
        "simulated_probability": [benchmark_sim],
        "exact_probability": [benchmark_exact],
        "stated_book_benchmark": [0.27987],
        "simulation_minus_exact": [benchmark_sim - benchmark_exact],
        "approx_95pct_mc_half_width": [benchmark_half_width],
    }
)

benchmark.style.format(
    {
        "scoring_probability": "{:.3%}",
        "simulated_probability": "{:.4%}",
        "exact_probability": "{:.6%}",
        "stated_book_benchmark": "{:.3%}",
        "simulation_minus_exact": "{:+.4%}",
        "approx_95pct_mc_half_width": "{:.4%}",
    }
)

In [ ]:
assert round(benchmark_exact * 100, 3) == 27.987
assert abs(benchmark_sim - benchmark_exact) <= 3 * benchmark_se

print(f"Exact result at p=72.5%: {benchmark_exact:.9%}")
print(f"Simulation estimate:       {benchmark_sim:.9%}")
print(f"Absolute error:            {abs(benchmark_sim - benchmark_exact):.6%}")

The benchmark passes if the independently calculated exact value rounds to 27.987% and the Monte Carlo estimate is within three simulation standard errors of that exact value.

## How does scoring probability change the chance of sudden death?

Keep everything else fixed and vary only `p`. Because the model is symmetric between scoring and missing, the probability at `p` is the same as at `1-p`. For football-relevant interpretation we therefore show the upper half of the range, from 50% to 95% conversion.

We simulate 500,000 shootouts at each five-percentage-point step and compare those estimates with the exact curve.

In [ ]:
sweep_p = np.arange(0.50, 0.951, 0.05)
sweep_n = 500_000
sweep_rng = np.random.default_rng(SEED + 1)

rows = []
for p in sweep_p:
    simulated = simulate_sudden_death_probability(float(p), sweep_n, sweep_rng)
    exact = exact_sudden_death_probability(float(p))
    rows.append(
        {
            "scoring_probability": p,
            "simulated_probability": simulated,
            "exact_probability": exact,
            "simulation_minus_exact": simulated - exact,
        }
    )

sweep = pd.DataFrame(rows)
sweep.style.format(
    {
        "scoring_probability": "{:.0%}",
        "simulated_probability": "{:.3%}",
        "exact_probability": "{:.3%}",
        "simulation_minus_exact": "{:+.3%}",
    }
)

In [ ]:
curve_p = np.linspace(0.50, 0.95, 181)
curve_exact = np.array([exact_sudden_death_probability(float(p)) for p in curve_p])

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(curve_p, curve_exact, label="Exact probability")
ax.scatter(
    sweep["scoring_probability"],
    sweep["simulated_probability"],
    label="Simulation (500k per point)",
)
ax.axvline(benchmark_p, linestyle="--", linewidth=1, label="72.5% benchmark")
ax.set(
    title="Higher conversion rates make sudden death more likely",
    xlabel="Penalty scoring probability",
    ylabel="Probability of reaching sudden death",
)
ax.set_xlim(0.50, 0.95)
ax.set_ylim(0.20, 0.70)
ax.legend()
plt.show()

In [ ]:
selected_p = [0.50, 0.60, 0.70, 0.725, 0.80, 0.90, 0.95]
selected = pd.DataFrame(
    {
        "scoring_probability": selected_p,
        "exact_sudden_death_probability": [
            exact_sudden_death_probability(p) for p in selected_p
        ],
    }
)
selected.style.format(
    {
        "scoring_probability": "{:.1%}",
        "exact_sudden_death_probability": "{:.3%}",
    }
)

## What this first study shows

Within this equal-team, independent, constant-`p` model:

- The 72.5% benchmark is reproduced independently: the exact probability of reaching sudden death is about **27.987%**.
- The simulation agrees with the exact result to ordinary Monte Carlo error.
- At `p = 50%`, the exact sudden-death probability is **24.609%**. This is the minimum of the symmetric curve.
- Once conversion probability is above 50%, **better penalty conversion makes sudden death more likely**, not less likely. At high conversion rates both teams increasingly cluster on the same high five-kick totals, especially 5–5.
- The effect is modest around ordinary middle-high probabilities, then becomes much stronger as conversion approaches certainty.

This answers the deliberately narrow first question. We should **not** yet add heterogeneous kickers, team-strength differences, first-kicker effects, score-state pressure or sudden-death duration. Those are separate model changes and should only be introduced if a later sporting question requires them.

## Limitations

This is a baseline model, not a claim that real penalties are independent or identically distributed. It deliberately ignores known sources of heterogeneity and any psychological or strategic effects. Its value here is that the assumptions are explicit, the simulation is reproducible, and the result can be checked against an exact probability.